XGBOOST DATASET 1 DEMOSU,DİĞER 2 DEMODAKİ EFEKTİF YÖNTEMLERİ KULLLANMAYA DEVAM

In [2]:
import numpy as np
import pandas as pd
import joblib


from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
%run MLProject.ipynb
df = pd.read_csv('usgs_main.csv')
df1 = df.copy()
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1.sort_values('time')
df1 = df1.dropna(subset=['latitude','longitude','depth','mag','time'])

dfweek_xgb = df1.set_index('time').resample('W').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
    'magType': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'status': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'net': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
})
dfweek_xgb = dfweek_xgb.reset_index(drop=True)
dfweek_xgb.index = dfweek_xgb.index + 1
dfweek_xgb.index.name = 'timeindex'

dfweek_xgb['futuremag'] = dfweek_xgb['mag'].shift(-1)
dfweek_xgb['futuredepth'] = dfweek_xgb['depth'].shift(-1)
dfweek_xgb['futurelat'] = dfweek_xgb['latitude'].shift(-1)
dfweek_xgb['futurelon'] = dfweek_xgb['longitude'].shift(-1)
dfweek_xgb = dfweek_xgb.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])
train_xgb = dfweek_xgb[:int((4*len(dfweek_xgb))/5)]
test_xgb = dfweek_xgb[int(4*len(dfweek_xgb)/5):]

In [3]:
best_model_xgb_1 = joblib.load('models/xgboost_dataset1.pkl')
test_predictions_xgb = best_model_xgb_1.predict(test_xgb)
test_actual_xgb = test_xgb[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_xgb = min(len(test_predictions_xgb), len(test_actual_xgb))
target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

In [4]:
print("DATASET 1 XGBoost SONUÇLARI:")
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_xgb.shape[1] and i < test_actual_xgb.shape[1]:
        target_mse = mean_squared_error(
            test_actual_xgb.iloc[:min_len_xgb, i], 
            test_predictions_xgb[:min_len_xgb, i]
        )
        target_mae = mean_absolute_error(
            test_actual_xgb.iloc[:min_len_xgb, i], 
            test_predictions_xgb[:min_len_xgb, i]
        )
        target_r2 = r2_score(
            test_actual_xgb.iloc[:min_len_xgb, i], 
            test_predictions_xgb[:min_len_xgb, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

DATASET 1 XGBoost SONUÇLARI:
   Magnitude   : MSE=0.058, MAE=0.217, R²=-5.052
   Depth       : MSE=14.691, MAE=3.054, R²=-1.431
   Latitude    : MSE=1.559, MAE=0.982, R²=-0.012
   Longitude   : MSE=18.600, MAE=3.145, R²=-0.013


In [5]:
magnitude_thresholds = [2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    predicted_magnitudes = test_predictions_xgb[:min_len_xgb, 0]
    actual_magnitudes = test_actual_xgb.iloc[:min_len_xgb, 0].values
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"\nBüyüklük Eşiği: {threshold}")
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=2.0): 0
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 3.5
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=3.5): 0
Tahmin edilen deprem sayısı (>=3.5): 0
Doğru tahmin sayısı: 7
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 4.0
Toplam örnek sayısı: 7
Gerçek deprem sayısı (>=4.0): 0
Tahmin edilen deprem sayısı (>=4.0): 0
Doğru tahmin sayısı: 

c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

NORMAL XG BOOST TESTİNDE KULLANILAN METRİKLERLE TEST PERFORMANSLARINA ERİŞİM

In [6]:
test_with_predictions_xgb = test_xgb.iloc[:min_len_xgb].copy()
test_with_predictions_xgb['predicted_mag'] = test_predictions_xgb[:min_len_xgb, 0]
test_with_predictions_xgb['predicted_lat'] = test_predictions_xgb[:min_len_xgb, 2]
test_with_predictions_xgb['predicted_lon'] = test_predictions_xgb[:min_len_xgb, 3]

test_with_predictions_xgb['lat_group'] = np.round(test_with_predictions_xgb['latitude'])
test_with_predictions_xgb['lon_group'] = np.round(test_with_predictions_xgb['longitude'])
test_with_predictions_xgb['location_group'] = test_with_predictions_xgb['lat_group'].astype(str) + '_' + test_with_predictions_xgb['lon_group'].astype(str)

location_groups_xgb = test_with_predictions_xgb.groupby('location_group').size()
valid_locations_xgb = location_groups_xgb[location_groups_xgb >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_xgb)}")

threshold = 1.5

for location in valid_locations_xgb[:10]:
    location_data = test_with_predictions_xgb[test_with_predictions_xgb['location_group'] == location]
    lat, lon = location.split('_')
    
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()


Yeterli veri olan bölge sayısı: 7
Bölge (37.0°, -108.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.78
  Max tahmin büyüklük: 1.45

Bölge (37.0°, -112.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Max gerçek büyüklük: 1.60
  Max tahmin büyüklük: 1.45

Bölge (37.0°, -114.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.66
  Max tahmin büyüklük: 1.54

Bölge (37.0°, -115.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.68
  Max tahmin büyüklük: 1.56

Bölge (37.0°, -117.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.85
  Max tahmin büyüklük: 1.53

Bölge (38.0°, -113.0°) - Veri sayısı: 1
  Gerçek: Deprem var
  Tahmin: Deprem var
  Doğru tahmin: ✓
  Max gerçek büyüklük: 1.72
  Max tahmin büyüklük: 1.59

Bölge (40.0°, -118.0°) -

DATASET2 DEMO

In [7]:
pn = pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
df2 = pn.copy()
df2 = df2.rename(columns={
    'Time':'time', 'Mag':'mag', 'Depth':'depth', 
    'Latitude':'latitude', 'Longitude':'longitude'
})

df2['time'] = pd.to_datetime(df2['time'])
df2 = df2.sort_values('time')
df2 = df2.dropna(subset=['latitude','longitude','depth','mag','time'])

dfyear_xgb = df2.set_index('time').resample('YE').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
    'MagType': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown',
    'Type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown',
    'status': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown',
    'net': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 and len(x) > 0 else 'Unknown'
})

dfyear_xgb = dfyear_xgb.rename(columns={'MagType':'magType','Type':'type'})

dfyear_xgb = dfyear_xgb.reset_index(drop=True)
dfyear_xgb.index = dfyear_xgb.index + 1
dfyear_xgb.index.name = 'timeindex'

dfyear_xgb['futuremag'] = dfyear_xgb['mag'].shift(-1)
dfyear_xgb['futuredepth'] = dfyear_xgb['depth'].shift(-1)
dfyear_xgb['futurelat'] = dfyear_xgb['latitude'].shift(-1)
dfyear_xgb['futurelon'] = dfyear_xgb['longitude'].shift(-1)
dfyear_xgb = dfyear_xgb.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])

train_xgb_2 = dfyear_xgb[:int((4*len(dfyear_xgb))/5)]
test_xgb_2 = dfyear_xgb[int(4*len(dfyear_xgb)/5):]

In [8]:
best_model_xgb_2 = joblib.load('models/xgboost_dataset2.pkl')
test_predictions_xgb_2 = best_model_xgb_2.predict(test_xgb_2)
test_actual_xgb_2 = test_xgb_2[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_xgb_2 = min(len(test_predictions_xgb_2), len(test_actual_xgb_2))

AYNI METRİK BLOKLARIYLA ÖLÇÜMLER

In [9]:
for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_xgb_2.shape[1] and i < test_actual_xgb_2.shape[1]:
        target_mse = mean_squared_error(
            test_actual_xgb_2.iloc[:min_len_xgb_2, i], 
            test_predictions_xgb_2[:min_len_xgb_2, i]
        )
        target_mae = mean_absolute_error(
            test_actual_xgb_2.iloc[:min_len_xgb_2, i], 
            test_predictions_xgb_2[:min_len_xgb_2, i]
        )
        target_r2 = r2_score(
            test_actual_xgb_2.iloc[:min_len_xgb_2, i], 
            test_predictions_xgb_2[:min_len_xgb_2, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

   Magnitude   : MSE=0.001, MAE=0.022, R²=-1.111
   Depth       : MSE=155.373, MAE=10.910, R²=-0.377
   Latitude    : MSE=16.833, MAE=3.190, R²=-0.443
   Longitude   : MSE=202.669, MAE=11.954, R²=-0.145


In [10]:
magnitude_thresholds_2 = [6.0, 6.5, 7.0, 7.5, 8.0]

for threshold in magnitude_thresholds_2:
    predicted_magnitudes = test_predictions_xgb_2[:min_len_xgb_2, 0]
    actual_magnitudes = test_actual_xgb_2.iloc[:min_len_xgb_2, 0].values
    
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"\nBüyüklük Eşiği: {threshold}")
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")



Büyüklük Eşiği: 6.0
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=6.0): 0
Tahmin edilen deprem sayısı (>=6.0): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 6.5
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=6.5): 0
Tahmin edilen deprem sayısı (>=6.5): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.0
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=7.0): 0
Tahmin edilen deprem sayısı (>=7.0): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 7.5
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=7.5): 0
Tahmin edilen deprem sayısı (>=7.5): 0
Doğru tahmin sayısı: 21
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

Büyüklük Eşiği: 8.0
Toplam örnek sayısı: 21
Gerçek deprem sayısı (>=8.0): 0
Tahmin edilen deprem sayısı (>=8.0): 0
Doğru tahmin

c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\eren1\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct s

In [11]:
test_with_predictions_xgb_2 = test_xgb_2.iloc[:min_len_xgb_2].copy()
test_with_predictions_xgb_2['predicted_mag'] = test_predictions_xgb_2[:min_len_xgb_2, 0]
test_with_predictions_xgb_2['predicted_lat'] = test_predictions_xgb_2[:min_len_xgb_2, 2]
test_with_predictions_xgb_2['predicted_lon'] = test_predictions_xgb_2[:min_len_xgb_2, 3]

test_with_predictions_xgb_2['lat_group'] = np.round(test_with_predictions_xgb_2['latitude'] / 5.0) * 5.0
test_with_predictions_xgb_2['lon_group'] = np.round(test_with_predictions_xgb_2['longitude'] / 5.0) * 5.0
test_with_predictions_xgb_2['location_group'] = test_with_predictions_xgb_2['lat_group'].astype(str) + '_' + test_with_predictions_xgb_2['lon_group'].astype(str)

location_groups_xgb_2 = test_with_predictions_xgb_2.groupby('location_group').size()
valid_locations_xgb_2 = location_groups_xgb_2[location_groups_xgb_2 >= 1].index

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations_xgb_2)}")

threshold = 6.0

for location in valid_locations_xgb_2:
    location_data = test_with_predictions_xgb_2[test_with_predictions_xgb_2['location_group'] == location]
    lat, lon = location.split('_')
    
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Max gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Max tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()

Yeterli veri olan bölge sayısı: 17
Bölge (-0.0°, 25.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.87
  Max tahmin büyüklük: 5.90

Bölge (-0.0°, 30.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.90
  Max tahmin büyüklük: 5.91

Bölge (-0.0°, 40.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.86
  Max tahmin büyüklük: 5.89

Bölge (-0.0°, 45.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.90

Bölge (-0.0°, 55.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.88
  Max tahmin büyüklük: 5.87

Bölge (-5.0°, 10.0°) - Veri sayısı: 1
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Max gerçek büyüklük: 5.89
  Max tahmin büyüklük: 5.90

Bölge (-5.0°, 20.0°) - Veri sayısı:

Yine tek bir veri için tahmin,RFDemodaki gibi 

In [12]:
def single_prediction_xgb(test_data, model, sample_index=5):
    if sample_index >= len(test_data):
        return None, None
    
    shiftnum = getattr(model, 'shiftnum', 3)#bu göz önüne alınmazsa veri tahmin için geçmişe bakmıyor ve boyut hatası oluşuyor.
    start_index = max(0, sample_index - shiftnum)
    end_index = sample_index + 1
    
    if sample_index < shiftnum:
        return None, None
    
    sample_rows = test_data.iloc[start_index:end_index]
    target_row = test_data.iloc[sample_index:sample_index+1]
    
    feature_cols = ['mag', 'depth', 'latitude', 'longitude']
    for col in feature_cols:
        if col in target_row.columns:
            print(f"  {col:12}: {target_row[col].iloc[0]:.4f}")
    
    target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
    for col in target_cols:
        if col in target_row.columns:
            print(f"  {col:12}: {target_row[col].iloc[0]:.4f}")
    
    try:
        prediction = model.predict(sample_rows)
        
        if len(prediction) > 0:
            final_prediction = prediction[-1]
            
            print(f"\n TAHMİNLER VE HATALAR:")
            target_labels = ['Future Mag', 'Future Depth', 'Future Lat', 'Future Lon']
            
            for i, (col, label) in enumerate(zip(target_cols, target_labels)):
                if col in target_row.columns and i < len(final_prediction):
                    actual_val = target_row[col].iloc[0]
                    pred_val = final_prediction[i]
                    error = abs(actual_val - pred_val)
                    
                    print(f"  {label:12}: Tahmin={pred_val:.4f}, Gerçek={actual_val:.4f}, Hata={error:.4f}")
            
            return final_prediction.reshape(1, -1), target_row[target_cols].values
        else:
            print("Model tahmin döndürmedi!")
            return None, None
            
    except Exception as e:
        print(f"Hata: {str(e)}")
        return None, None

print("DATASET 1 XGBoost TESTİ:")
tahmin_xgb1, gercek_xgb1 = single_prediction_xgb(test_xgb, best_model_xgb_1, sample_index=5)
digertahmin_xgb1,digergercek_xgb1=single_prediction_xgb(test_xgb,best_model_xgb_1,sample_index=8)
print("\nDATASET 2 XGBoost TESTİ:")
tahmin_xgb2, gercek_xgb2 = single_prediction_xgb(test_xgb_2, best_model_xgb_2, sample_index=5)
digertahmin_xgb2,digergercek_xgb2=single_prediction_xgb(test_xgb_2,best_model_xgb_2,sample_index=8)

DATASET 1 XGBoost TESTİ:
  mag         : 1.5968
  depth       : 20.3829
  latitude    : 39.6866
  longitude   : -117.9273
  futuremag   : 1.8904
  futuredepth : 23.0301
  futurelat   : 36.5783
  futurelon   : -108.0801

 TAHMİNLER VE HATALAR:
  Future Mag  : Tahmin=1.5261, Gerçek=1.8904, Hata=0.3643
  Future Depth: Tahmin=19.1375, Gerçek=23.0301, Hata=3.8926
  Future Lat  : Tahmin=37.3171, Gerçek=36.5783, Hata=0.7388
  Future Lon  : Tahmin=-114.5682, Gerçek=-108.0801, Hata=6.4882
  mag         : 1.7586
  depth       : 21.3244
  latitude    : 37.2342
  longitude   : -114.4885
  futuremag   : 1.9865
  futuredepth : 30.3099
  futurelat   : 34.0627
  futurelon   : -103.3797

 TAHMİNLER VE HATALAR:
  Future Mag  : Tahmin=1.4538, Gerçek=1.9865, Hata=0.5327
  Future Depth: Tahmin=18.7204, Gerçek=30.3099, Hata=11.5896
  Future Lat  : Tahmin=37.2327, Gerçek=34.0627, Hata=3.1699
  Future Lon  : Tahmin=-112.8818, Gerçek=-103.3797, Hata=9.5021

DATASET 2 XGBoost TESTİ:
  mag         : 5.8511
  dep

Yine Klasör Okuma Kodu,RF Demoya benziyor

In [13]:
import os
import glob

def create_sample_folder_xgb(test_data, folder_name="demo_samples_xgb", n_samples=15, shiftnum=3):
    os.makedirs(folder_name, exist_ok=True)
    
    print(f"{folder_name} klasörü oluşturuluyor...")
    
    max_start_idx = len(test_data) - shiftnum - 1
    np.random.seed(42)
    sample_count = min(n_samples, max_start_idx)
    start_indices = np.random.choice(max_start_idx, size=sample_count, replace=False)
    
    for i, start_idx in enumerate(start_indices):
        end_idx = start_idx + shiftnum + 1
        sample_chunk = test_data.iloc[start_idx:end_idx]
        
        filename = f"{folder_name}/sample_{i+1:02d}.csv"
        sample_chunk.to_csv(filename, index=False)
    
    print(f"{len(start_indices)} adet örnek {folder_name} klasörüne kaydedildi.")
    print(f"Her dosya {shiftnum + 1} satır içeriyor.")
    return folder_name

def predict_from_folder_xgb(folder_name, model, shiftnum=3):
    csv_files = glob.glob(f"{folder_name}/*.csv")
    csv_files.sort()
    
    all_predictions = []
    all_actuals = []
    all_filenames = []
    
    print(f"\n{folder_name} klasöründeki {len(csv_files)} dosya işleniyor...")
    
    for csv_file in csv_files:
        try:
            sample_df = pd.read_csv(csv_file)
            if len(sample_df) < shiftnum + 1:
                continue
            target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
            available_targets = [col for col in target_cols if col in sample_df.columns]
            if len(available_targets) == 0:
                continue
            
            actual_values = sample_df[available_targets].iloc[-1].values
            
            try:
                prediction = model.predict(sample_df)
                
                if len(prediction) > 0:
                    final_prediction = prediction[-1]
                    all_predictions.append(final_prediction)
                    all_actuals.append(actual_values)
                    all_filenames.append(os.path.basename(csv_file))
                else:
                    print(f"{os.path.basename(csv_file)}: Model tahmin döndürmedi")
                    
            except Exception as pred_error:
                print(f"{os.path.basename(csv_file)}: Tahmin hatası - {str(pred_error)}")
            
        except Exception as e:
            print(f"{csv_file}: Dosya okuma hatası - {str(e)}")
    
    if len(all_predictions) > 0:
        return np.array(all_predictions), np.array(all_actuals), all_filenames
    else:
        return np.array([]), np.array([]), []
def run_demo_xgb(test_data, model, model_name="XGBoost", n_samples=15):
    shiftnum = getattr(model, 'shiftnum', 3)
    
    folder_name = f"demo_samples_xgb_{model_name.lower().replace(' ', '_')}"
    create_sample_folder_xgb(test_data, folder_name, n_samples, shiftnum)
    
    predictions, actuals, filenames = predict_from_folder_xgb(folder_name, model, shiftnum)
    
    if len(predictions) > 0:
        target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
        
        for i, label in enumerate(target_labels):
            if i < predictions.shape[1] and i < actuals.shape[1]:
                mse = mean_squared_error(actuals[:, i], predictions[:, i])
                mae = mean_absolute_error(actuals[:, i], predictions[:, i])
                r2 = r2_score(actuals[:, i], predictions[:, i])
                
                print(f"{label:<12}: MSE={mse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")
    
    return predictions, actuals, filenames

def predict_single_file_xgb(file_path, model):
    print("tek bir file için XGBoost tahmini")
    
    try:
        sample_df = pd.read_csv(file_path)
        print(f"Dosya boyutu: {len(sample_df)} satır")
        
        shiftnum = getattr(model, 'shiftnum', 3)
        
        if len(sample_df) < shiftnum + 1:
            return None, None
        
        last_row = sample_df.iloc[-1]
        print(f"  Mevcut Büyüklük: {last_row['mag']:.3f}")
        print(f"  Mevcut Derinlik: {last_row['depth']:.3f}")
        print(f"  Mevcut Konum: ({last_row['latitude']:.3f}, {last_row['longitude']:.3f})")
        
        target_cols = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
        available_targets = [col for col in target_cols if col in sample_df.columns]
        
        if len(available_targets) == 0:
            return None, None
        
        print(f"\nHEDEF DEĞERLER:")
        actual_values = []
        for col in available_targets:
            val = last_row[col]
            actual_values.append(val)
            col_name = col.replace('future', '').title()
            print(f"  Gelecek {col_name}: {val:.3f}")
        
        prediction = model.predict(sample_df)
        
        if len(prediction) > 0:
            final_prediction = prediction[-1]
            
            print(f"\n XGBoost MODEL TAHMİNLERİ:")
            target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']
            
            for i, (col, label) in enumerate(zip(available_targets, target_labels)):
                if i < len(final_prediction) and i < len(actual_values):
                    pred_val = final_prediction[i]
                    actual_val = actual_values[i]
                    error = abs(actual_val - pred_val)
                    
                    print(f"  {label:<12}: Tahmin={pred_val:.3f}, Gerçek={actual_val:.3f}, Hata={error:.3f}")
            
            return final_prediction, np.array(actual_values)
        else:
            print("Model tahmin döndürmedi!")
            return None, None
            
    except Exception as e:
        print(f"\n Hata: {str(e)}")
        return None, None

print("\n\nKLASÖR DEMO TESTLERİ:")
print("DATASET 1 - XGBoost Model Demo")
pred_xgb1, act_xgb1, files_xgb1 = run_demo_xgb(
    test_xgb, best_model_xgb_1, "XGBoost_Dataset1", 10
)

# İlk oluşturulan klasörden bir dosya seç ve test et
sample_file = "demo_samples_xgb_xgboost_dataset1/sample_01.csv"
if os.path.exists(sample_file):
    pred_single_xgb, act_single_xgb = predict_single_file_xgb(sample_file, best_model_xgb_1)
else:
    print(f"Örnek dosya bulunamadı: {sample_file}")

# Dataset 2 için demo
print("\nDATASET 2 - XGBoost Model Demo")
pred_xgb2, act_xgb2, files_xgb2 = run_demo_xgb(
    test_xgb_2, best_model_xgb_2, "XGBoost_Dataset2", 8
)

# Dataset 2 için tek dosya testi
print("\n Dataset 2 - Klasörden tek dosya testi:")
sample_file_2 = "demo_samples_xgb_xgboost_dataset2/sample_01.csv"
if os.path.exists(sample_file_2):
    pred_single_xgb2, act_single_xgb2 = predict_single_file_xgb(sample_file_2, best_model_xgb_2)
else:
    print(f" Örnek dosya bulunamadı: {sample_file_2}")    



KLASÖR DEMO TESTLERİ:
DATASET 1 - XGBoost Model Demo
demo_samples_xgb_xgboost_dataset1 klasörü oluşturuluyor...
6 adet örnek demo_samples_xgb_xgboost_dataset1 klasörüne kaydedildi.
Her dosya 3 satır içeriyor.

demo_samples_xgb_xgboost_dataset1 klasöründeki 6 dosya işleniyor...
Magnitude   : MSE=0.061, MAE=0.222, R²=-4.944
Depth       : MSE=14.883, MAE=2.835, R²=-1.092
Latitude    : MSE=2.157, MAE=1.188, R²=-0.241
Longitude   : MSE=20.904, MAE=3.796, R²=-0.027
tek bir file için XGBoost tahmini
Dosya boyutu: 3 satır
  Mevcut Büyüklük: 1.720
  Mevcut Derinlik: 21.887
  Mevcut Konum: (37.446, -113.507)

HEDEF DEĞERLER:
  Gelecek Mag: 1.659
  Gelecek Depth: 18.975
  Gelecek Lat: 37.308
  Gelecek Lon: -116.806

 XGBoost MODEL TAHMİNLERİ:
  Magnitude   : Tahmin=1.561, Gerçek=1.659, Hata=0.098
  Depth       : Tahmin=19.752, Gerçek=18.975, Hata=0.776
  Latitude    : Tahmin=37.496, Gerçek=37.308, Hata=0.188
  Longitude   : Tahmin=-113.052, Gerçek=-116.806, Hata=3.754

DATASET 2 - XGBoost Model